# N028 · 固定长度滑动窗口

**目标：** 通过移出移入更新固定窗口统计量。

**先修：** N020, N025。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 窗口长度；增量更新；频次向量；零长度契约。

**配套讲解来源：** [同名逐章意见](../../comment/028_fixed_windows.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个"固定长度窗口扫一遍"的问题，共同心法是：窗口长度 k 不变，每次窗口右移一格时，**不重算整个窗口，只减去刚离开的、加上刚进来的**。

**问题一（定长最大平均值，`max_average`）**：给你一个数组和长度 k，找出长度恰为 k 的连续子数组里平均值最大的那个，返回这个平均值。举个具体到数字的例子：输入 `[1,12,-5,-6,50,3]`、k=4 → 输出 `12.75` → 为什么？所有长度 4 的窗口有三个：`[1,12,-5,-6]` 和是 2、`[12,-5,-6,50]` 和是 51、`[-5,-6,50,3]` 和是 42，最大和 51 除以 4 得 12.75。暴力做法每开一个新窗口就把 k 个数重新加一遍，代价 O(n·k)；本章把"重算 k 项"变成"更新两项"（减去左端离开的、加上右端进来的），整个算法 O(n)。

**问题二（字符串包含排列，`check_inclusion`）**：给你短串 s1 和长串 s2，问 s2 里有没有一段长度恰为 `len(s1)` 的子串，其字符组成是 s1 的**排列**（同样的字符、同样的个数、顺序无所谓）。例如输入 s1=`'ab'`、s2=`'eidbaooo'` → 输出 `True` → 为什么？s2 的下标 3、4 位置是 `'ba'`，正好是 ab 的排列；而 s2=`'eidboaoo'` 里所有长度 2 的子串（ei、id、db、bo、oa、ao、oo）没有一个恰含一个 a 和一个 b，输出 `False`。做法是把"两个多重集合相同"翻译成"逐字符计数完全一致"，在滑动窗口里增量维护 s2 当前窗口的计数表，每步与 s1 的计数表整体比较。

## 2. 基础知识：先读懂这些词

- **固定长度滑动窗口（fixed-size sliding window）**：一段连续区间 [left, right]，长度恒为 k，每一步整体右移一格——left 和 right 同时 +1。适合"所有长度为 k 的子区间"类问题。
- **增量更新（incremental update）**：本章的灵魂。窗口右移时聚合量（和、计数表）的变化只有两处：减去 `nums[right-k]`（刚滑出左端的旧元素）、加上 `nums[right]`（刚滑入右端的新元素）。窗口和从"重算 k 项"降到"更新两项"，这是滑动窗口省时间的全部秘密。
- **频次向量（frequency vector）**：把"一个串里每个字符出现几次"记成一张表。两张频次表完全相同，当且仅当两个串是彼此的排列（字符多重集合相同）——因为顺序信息被扔掉后，剩下的全部信息就是"每种字符有几个"。
- **多重集合（multiset）**：允许元素重复的集合，"两个 a 和一个 b"与"一个 b 和两个 a"是同一个多重集合。字符串的排列关系就是多重集合相等。
- **Counter（计数字典）**：Python 标准库 `collections.Counter`，传入字符串自动生成"字符→出现次数"的字典，比如 `Counter('aab')` 得到 `{'a':2,'b':1}`。两个 Counter 用 `==` 比较即判断两张频次表是否逐项相同。
- **零长度契约（empty-pattern contract）**：边界情形的约定。s1 为空串时"空排列在任何串里"都成立，返回 True；而 `max_average` 对非法的 k（小于 1 或大于数组长度）则直接抛 ValueError 拒绝服务——两种边界一个宽容一个严格，都是"把丑话说在前面"的契约设计。
- **零项清理（删除计数为 0 的键）**：窗口滑走某个字符后它的计数可能变成 0，代码当场把这个键从字典里删掉。这样 `have` 字典里留下的永远是"当前窗口里真实存在的字符"，`have==need` 的比较结果才精确对应"当前窗口多重集合等于 s1 多重集合"，不会残留僵尸键干扰判断。

## 3. 思路推导：从小例子开始

**`max_average` 的思路：**

- **Step 1：验证 k 合法。** k 必须满足 1 ≤ k ≤ len(nums)，否则没有长度为 k 的窗口可言，直接报错。
- **Step 2：算出第一个窗口的和。** 前 k 个数直接求和，作为 total 和 best 的初值。
- **Step 3：窗口逐格右移，两项更新。** right 从 k 走到末尾，每步 `total += 新进来的 − 刚离开的`（刚离开的就是 `nums[right-k]`，即新窗口左端的前一格）。
- **Step 4：每步更新最优，最后除以 k。** best 记录窗口和的最大值，返回 `best/k`——k 个数里和最大等价于平均值最大，不必每步做除法。

**为什么要"记和、不记平均值"？** k 是常数，和与平均值一一对应、同时取最大；但整数加减没有误差、浮点除法有，把 n−k+1 次除法合并成最后 1 次，既省了计算也让中间比较全部发生在整数域上，这是数值上更稳的写法。

**手算演示：`max_average([1,12,-5,-6,50,3], k=4)`（“运行示例”部分 表格的完整对照）**

| 阶段 | 窗口 | 计算 | total | best |
|---|---|---|---|---|
| 初始化 | [1,12,-5,-6] | 1+12−5−6 | 2 | 2 |
| right=4 | [12,-5,-6,50] | total+50−nums[0]=2+50−1 | 51 | 51 |
| right=5 | [-5,-6,50,3] | total+3−nums[1]=51+3−12 | 42 | 51 |

返回 51/4 = 12.75。“运行示例”部分 的表格把三个窗口的左端、内容、和、平均值全列了出来（0.5、12.75、10.5 三行），与我们手算的三个 total 完全一致——你看每步真正的算术只有"加一个数、减一个数"两次。

**`check_inclusion` 的思路：**

- **Step 1：处理两个边界。** s1 为空返回 True（空排列随处可取）；s1 比 s2 长返回 False（塞不下）。
- **Step 2：建两张计数表。** need 是 s1 的频次表（目标），have 是 s2 前 k 个字符的频次表（第一个窗口）。
- **Step 3：窗口右移，增量维护 have。** 新字符计数 +1，滑出的旧字符计数 −1；旧字符计数归零就把这个键删掉。
- **Step 4：每步比较两张表。** 相等说明当前窗口是 s1 的排列，返回 True；滑完仍不等返回 False。

对比两题的增量公式可以加深记忆：数值窗口是 `total ± 一个数`（加新减旧，聚合量是标量）；频次窗口是 `have[字符] ± 1`（新字符进账、旧字符销账，聚合量是整张表）。形式不同，骨架完全一样。

**手算演示：`check_inclusion('ab','eidbaooo')`**

| 步 | 窗口 | have 变化 | have | 等于 need={a:1,b:1}? |
|---|---|---|---|---|
| 初始 | ei | — | {e:1,i:1} | 否 |
| right=2 | id | +d −e | {i:1,d:1} | 否 |
| right=3 | db | +b −i | {d:1,b:1} | 否 |
| right=4 | **ba** | +a −d | {b:1,a:1} | **是 → 返回 True** |

再看反例 `check_inclusion('ab','eidboaoo')`：窗口依次是 ei、id、db、bo、oa、ao、oo，have 从未同时含一个 a 和一个 b（oa 窗口有 a 没 b，bo 窗口有 b 没 a），扫完返回 False。两例合看能体会"频次表相等"这个判据怎么工作：它不管顺序，只看每种字符的个数是否严丝合缝。

## 4. 核心代码：max_average

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def max_average(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('requires 1 <= k <= len(nums)')
    total = sum(nums[:k])
    best = total
    for right in range(k, len(nums)):
        total += nums[right] - nums[right - k]
        best = max(best, total)
    return best / k
```

### 逐段读懂代码

### 函数 1：`max_average`（定长最大平均值）

先贴 notebook “分段实现”部分 里的真实代码（含文件顶部的 import）：

```python
from collections import Counter

def max_average(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('requires 1 <= k <= len(nums)')
    total = sum(nums[:k])
    best = total
    for right in range(k, len(nums)):
        total += nums[right] - nums[right - k]
        best = max(best, total)
    return best / k
```

- **第 1 行 `from collections import Counter`**：这是文件级导入，第二个函数要用；放在顶部一次导入。
- **第 3 行 `if not 1<=k<=len(nums): raise ValueError(...)`**：参数闸门。`1<=k<=len(nums)` 是 Python 的链式比较，"not" 取反后即"k 越界"。k<1 时窗口无意义，k 大于数组长度时窗口根本放不下，两种情况都抛 ValueError 并附上人话提示，而不是默默返回错误答案。这就是"零长度契约"里严格的那一半。
- **第 4 行 `total=sum(nums[:k]); best=total`**：第一个窗口（下标 0..k−1）直接求和初始化 total，best 一开始就等于它——只有一个窗口时循环不进，答案也不至于没初值。
- **第 5–6 行滑动主体**：`for right in range(k,len(nums))` 让右端从 k 走到 n−1，每轮新窗口是 [right−k+1, right]。核心一行 `total+=nums[right]-nums[right-k]`：`nums[right]` 是刚滑入的右端新元素，`nums[right-k]` 是刚滑出的左端旧元素（新左端 left=right−k+1 的前一格），一加一减恰好把窗口和推进到新位置——两项更新替代 k 项重算。`best=max(best,total)` 一句话完成"比现在好就更新"。
- **第 8 行 `return best/k`**：best 是最大窗口和，除以固定的 k 得最大平均值。整个循环里只做整数加减、最后才做一次除法，既省事又避免每步除法带来的浮点累积误差。

### 函数 2：`check_inclusion`（字符串包含排列）

```python
def check_inclusion(s1, s2):
    k = len(s1)
    if k == 0:
        return True
    if k > len(s2):
        return False
    need = Counter(s1)
    have = Counter(s2[:k])
    if need == have:
        return True
    for right in range(k, len(s2)):
        have[s2[right]] += 1
        old = s2[right - k]
        have[old] -= 1
        if have[old] == 0:
            del have[old]
        if have == need:
            return True
    return False
```

- **第 2 行 `k=len(s1)`**：窗口长度就是短串长度——排列必须逐字符对上，长度差一都不行。
- **第 3 行 `if k==0: return True`**：空串的排列是空串，任何串（包括空串）都"包含"它，返回 True。这是零长度契约里宽容的那一半。
- **第 4 行 `if k>len(s2): return False`**：s2 比 s1 还短，塞不下任何长度 k 的窗口，返回 False。这行同时保证了后面 `s2[:k]` 切片不会制造"长度不足 k 的假窗口"。
- **第 5 行 `need=Counter(s1); have=Counter(s2[:k])`**：两张频次表就位——need 是永恒的目标，have 是当前窗口的实时状态。`s2[:k]` 是第一个窗口。
- **第 6 行 `if need==have: return True`**：第一个窗口可能就直接命中，别忘了先比一次再进循环。
- **第 8 行 `have[s2[right]]+=1`**：新右端字符计数 +1。Counter 访问不存在的键返回 0 而不报错，所以这行对首次出现的字符也安全。
- **第 9 行 `old=s2[right-k]; have[old]-=1`**：`old` 是刚滑出左端的字符（与 max_average 里的 `nums[right-k]` 同一位置逻辑），计数 −1。
- **第 10 行 `if have[old]==0: del have[old]`**：归零就删键。删完之后 have 里每个键的计数都是正数，字典内容恰好等于"当前窗口的多重集合"，第 11 行的比较因此干净可靠——不留零计数的僵尸键，两个多重集合是否相同一目了然。
- **第 11 行 `if have==need: return True`**：每滑一步比一次。两张表逐键相等意味着当前窗口字符组成与 s1 完全一致，命中。
- **第 13 行 `return False`**：滑完最后一个窗口仍未命中，s2 不包含 s1 的任何排列。

## 5. 分段实现：按顺序运行

**本章接口：** `max_average(nums, k)`、`check_inclusion(s1, s2)`

### 导入本章使用的库

In [1]:
from collections import Counter

### max_average

In [2]:
def max_average(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('requires 1 <= k <= len(nums)')
    total = sum(nums[:k])
    best = total
    for right in range(k, len(nums)):
        total += nums[right] - nums[right - k]
        best = max(best, total)
    return best / k

### check_inclusion

In [3]:
def check_inclusion(s1, s2):
    k = len(s1)
    if k == 0:
        return True
    if k > len(s2):
        return False
    need = Counter(s1)
    have = Counter(s2[:k])
    if need == have:
        return True
    for right in range(k, len(s2)):
        have[s2[right]] += 1
        old = s2[right - k]
        have[old] -= 1
        if have[old] == 0:
            del have[old]
        if have == need:
            return True
    return False

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,12,-5,-6,50,3]; k=4
show_table(['左端','窗口','窗口和','平均值'],[(l,a[l:l+k],sum(a[l:l+k]),sum(a[l:l+k])/k) for l in range(len(a)-k+1)])

左端,窗口,窗口和,平均值
0,"[1, 12, -5, -6]",2,0.5
1,"[12, -5, -6, 50]",51,12.75
2,"[-5, -6, 50, 3]",42,10.5


## 7. 正确性、适用条件与复杂度

窗口由原窗口删除恰好一个离开元素并加入恰好一个新元素，因此聚合量始终对应当前位置。相同长度的字符频次完全相同等价于排列关系。

**代价：** 数值窗口 O(n) 时间、O(1) 空间；字符版 O(nσ) 时间、O(σ) 空间，σ为比较的不同字符数；固定字母表视为 O(n)。

### 展开理解

**max_average 为什么是对的？** 关键在于"total 始终等于当前窗口的和"这件事从头到尾没破过：初始时 total 由第一个窗口直接求和而来，当然相等；此后每一步，窗口恰好发生两个变化——左端离开一个旧元素、右端进入一个新元素——代码恰好减去前者、加上后者，所以移动后的 total 精确等于新窗口的和。既然每个窗口的和都被算对，best 又是所有这些和的最大值，除以固定的 k 自然就是最大平均值。所有长度为 k 的窗口都被遍历（right 从 k 到 n−1 对应全部 n−k+1 个窗口），没有遗漏。

**check_inclusion 为什么是对的？** 先看判据本身：一个长度为 k 的窗口是 s1 的排列，当且仅当两边的"字符多重集合"相同，而这又当且仅当逐字符的频次表完全一致——因为频次表就是多重集合的完整账本，账本一样东西就一样。再看维护：与 max_average 同理，每步 have 恰好经历"+1 新字符、−1 旧字符"两个变化，加上零键清理，`have` 任何时刻都精确记录当前窗口的频次账本，所以每步的 `have==need` 比较都是对"当前窗口是否为排列"的如实判断。所有窗口都检查过后仍无命中，才能放心返回 False。两个边界（空 s1、s1 比 s2 长）也各得其所。

**复杂度是多少？** max_average 是 O(n) 时间：初始化一次 k 项求和，之后 n−k 次循环每次两次加减。n=10⁵、k=10⁴ 时，暴力是十亿次加法，本章做法只有约十万次加减——快一万倍，一瞬间完成。空间 O(1)：只有 total、best、right 几个变量。check_inclusion 是 O(n·σ)：滑动 n 步，每步的 `have==need` 字典比较要逐键核对，σ 是两张表里不同字符的种类数（“正确性与复杂度”部分 特意注明）；题目限定 26 个小写字母时 σ≤26 视为常数，总时间就是 O(n)。空间 O(σ) 存两张频次表，固定字母表下也是常数。教学实现故意用"整个 Counter 直接比较"来突出"频次表相等即排列"这个概念，工程上可换成维护"不匹配字符数"把每步比较降到 O(1)——但那是后话。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条看：

- `assert math.isclose(max_average([1,12,-5,-6,50,3],4),12.75)`：正常值，LeetCode 643 样例。因为答案是浮点数（51/4），用 `math.isclose` 比较而不是 `==`，避免二进制浮点表示的微小误差造成假失败——测的是算法本身，不是浮点的精确表示。
- `assert max_average([3,-1],1)==3`：k=1 边界。窗口退化成单个元素，答案就是最大元素 3；由于 3/1 恰好整除，浮点结果 3.0 与整数 3 相等成立，不需要 isclose。测的是"窗口最小宽度"下初始化路径正确（此时循环一次都不进，全靠初值撑住答案）。
- `assert max_average([3,-1],2)==1`：k 等于数组长度边界。唯一的窗口是整个数组，和为 2、平均 1；测的是"窗口最大宽度"时循环零次、直接用初值。
- `assert check_inclusion('ab','eidbaooo')`：正常 True 值，LeetCode 567 样例。'ba' 段命中，测的是滑动过程中的频次表比较能抓到中途窗口。
- `assert not check_inclusion('ab','eidboaoo')`：正常 False 值。a 和 b 都在但从未同窗，测的是"字符都在但窗口错开"时不会误报。
- `assert check_inclusion('','abc')`：零长度契约。空 s1 返回 True，测的是第 3 行那个专门分支。
- **双重循环对拍**：`product('ab',repeat=5)` 生成全部 32 个长度 5 的二字符字符串，对每个串和四种模式 p（'a'、'ab'、'aa'、'aba'）做对照——参考答案用暴力写法：枚举 s2 里所有长度 len(p) 的切片，看有没有一个切片排序后等于 p 排序（`any(sorted(s[i:i+len(p)])==sorted(p) for i in ...)`），断言函数值与暴力值相等。模式里 'aa' 考验重复字符计数、'aba' 考验长度 3 的频次对齐，32×4=128 种组合把滑动的各条路径都踩了一遍。特别值得学的是这条暴力的写法本身：`sorted(切片)==sorted(模式)` 用"排序后逐位相等"绕开了频次表，恰好从另一个角度定义了排列——教学实现与参考实现走的是两条不同的路，对拍才有意义。

全部通过后打印 `N028: 所有本章断言通过`。

In [5]:
import math

assert math.isclose(max_average([1, 12, -5, -6, 50, 3], 4), 12.75)

assert max_average([3, -1], 1) == 3

assert max_average([3, -1], 2) == 1

assert check_inclusion('ab', 'eidbaooo')

assert not check_inclusion('ab', 'eidboaoo')

assert check_inclusion('', 'abc')

from itertools import product

for s in map(''.join, product('ab', repeat=5)):
    for p in ['a', 'ab', 'aa', 'aba']:
        assert check_inclusion(p, s) == any((sorted(s[i:i + len(p)]) == sorted(p) for i in range(len(s) - len(p) + 1)))

print('N028: 所有本章断言通过')

N028: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 实现固定窗口元音计数。

**练习 2：** 比较重算窗口和增量更新。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（固定窗口元音计数）**：方向是把本章两套武器合到一题上——"窗口内有多少个元音字母"。提示：这不是频次表全等，而是"满足条件的字符计数和"这类可加统计量，思路更接近 max_average：维护一个计数变量，窗口右移时"新字符若是元音则 +1，滑出字符若是元音则 −1"。手算示例可用 `['l','e','e','t','c','o','d','e']`（LeetCode 1456 的背景）、k=3，先列出每个窗口的元音个数（lee=2、eet=2、etc=1、tco=1、cod=1、ode=2），最大 2。边界：k=1、全辅音串、k 等于串长。验证时写一个"逐窗口 sum(条件)"的暴力版对拍。
- **练习 2（比较重算窗口和增量更新）**：方向是量化"两项更新"的价值。提示：写两个版本——重算版每步 `sum(nums[left:left+k])`，增量版照抄本章；手算用 `[1,12,-5,-6,50,3]`、k=4 各列一张"每步做了几次加法"的账（重算版每步 4 次加法共 3 个窗口 12 次，增量版每步 2 次共 4 次）；再把 k 和 n 拉大（如 n=10⁴、k=10³）用 `%timeit` 或手数循环次数对比，得出 O(n·k) 对 O(n) 的直观感受。边界别忘 k=1（两者无差距）和 k=n（只有一个窗口），说明差距在 k 接近 n/2 时最悬殊。

两道练习做完你应该能自己说出滑动窗口的适用条件："窗口长度固定（或单调变化）、聚合量可由离开/进入的元素增量维护"。三个条件都满足，就值得把重算改成增量。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter

def max_average(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError('requires 1 <= k <= len(nums)')
    total = sum(nums[:k])
    best = total
    for right in range(k, len(nums)):
        total += nums[right] - nums[right - k]
        best = max(best, total)
    return best / k

def check_inclusion(s1, s2):
    k = len(s1)
    if k == 0:
        return True
    if k > len(s2):
        return False
    need = Counter(s1)
    have = Counter(s2[:k])
    if need == have:
        return True
    for right in range(k, len(s2)):
        have[s2[right]] += 1
        old = s2[right - k]
        have[old] -= 1
        if have[old] == 0:
            del have[old]
        if have == need:
            return True
    return False

# 示例与回归测试
import math

assert math.isclose(max_average([1, 12, -5, -6, 50, 3], 4), 12.75)

assert max_average([3, -1], 1) == 3

assert max_average([3, -1], 2) == 1

assert check_inclusion('ab', 'eidbaooo')

assert not check_inclusion('ab', 'eidboaoo')

assert check_inclusion('', 'abc')

from itertools import product

for s in map(''.join, product('ab', repeat=5)):
    for p in ['a', 'ab', 'aa', 'aba']:
        assert check_inclusion(p, s) == any((sorted(s[i:i + len(p)]) == sorted(p) for i in range(len(s) - len(p) + 1)))

print('N028: 所有本章断言通过')

N028: 所有本章断言通过


## 11. 代表题与迁移

- **643. Maximum Average Subarray I（子数组最大平均数 I）**：这题就是 `max_average` 本尊，练的是"窗口和的两项增量更新 + 最后统一除以 k"；注意它要求窗口长度恰为 k，答案返回浮点数。
- **567. Permutation in String（字符串的排列）**：这题就是 `check_inclusion` 本尊，练的是"频次表判排列 + 滑动时计数的加减与零键清理"；把"多重集合相等"翻译成"Counter 相等"是它的概念核心。

两题都标为 canonical（与本章一一对应）。迁移方向：643 推出"变长窗口求最大平均值 II"（需二分答案）、567 的频次表思路直接平移到 438（找异位词的所有起点），再往外就是 N029 起的变长窗口——固定窗口是变长窗口的零件，先把它拧熟练。自测问题：check_inclusion 里如果不做 `del have[old]` 的零键清理，`have==need` 在什么情况下会出问题？（提示：need 里没有的字符在 have 中留着 0 计数僵尸键时，字典比较可能判出 False——想想 'ab' 对 'ab' 滑过 'c' 窗口的情形。）

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。